## 4. RPN(Region Proposal Network)

RPN网络主要用于生成Proposal，即带有前景、背景信息和包围框信息的区域。RPN中，首先针对特征图生成若干Anchor(像素区域的锚框)，按照一定的规则对Anchor进行筛选、处理，并对通过Ground Truth包围框对进行训练，区分前景、背景以及包围框修正信息，进一步生成RoI，然后再对RoI处理得到Proposal。


### Anchor的生成规则

通过backbone网络，我们得到了5组尺寸不同、深度相同的特征图。在Mask R-CNN中，通过在特征图上生成不同尺寸的锚框(Anchor)，尝试框住特征图中的物体，下面我们来看一下Anchor的生成规则。

为了便于处理和计算，我们将输入图片长宽统一设置为1024x1024，那么，通过backbone网络输出的特征图尺寸如下：

| level |    size | depth | stride |
| ----- | ------- | ----- | ------ |
|    P2 | 256x256 |   256 |      4 |
|    P3 | 128x128 |   256 |      8 |
|    P4 |   64x64 |   256 |     16 |
|    P5 |   32x32 |   256 |     32 |
|    P5 |   16x16 |   256 |     64 |

其中，stride可以理解为相对于原图尺寸1024x1024的缩小倍数，例如，P2的尺寸为256x256，相对1024x1024在长宽上各缩小了4倍，stride记为4。同时，stride也是FPN上特征图一个像素相对于原始图像像素区域的边长，例如P3的stride为8，P3的一个像素，是由原图中8x8的像素区域得到的。

当特征图输入到RPN网络时，我们要对特征图上的每个像素生成3种长宽比的Anchor，即[1:2, 1:1, 2:1]，简单记为[0.5, 1, 2]。Anchor的面积不变，因此，假设比例为1的Anchor边长为$s$，那么，3种比例的Anchor长宽计算如下所示：


|  ratio |                 width |               height |
| ------ | --------------------- | -------------------- |
|    1:1 |                $$s$$  |                $$s$$ |
|    2:1 | $$s \times \sqrt{2}$$ | $$s\div \sqrt{2}$$   |
|    1:2 | $$s \div \sqrt{2} $$  | $$s\times \sqrt{2}$$ |

这样，就保证了比例，并且面积不变，都为$s^2$。我们在每个像素上都生成3个Anchor，则生成Anchor的数量为：

| level | calculation | anchors number |
| ----- | ----------- | -------------- | 
|    P2 |   256x256x3 |         196608 |
|    P3 |   128x128x3 |          49152 |
|    P4 |     64x64x3 |          12288 |
|    P5 |     32x32x3 |           3072 |
|    P5 |     16x16x3 |            768 |
|   all |           - |         261888 |

接下来，我们使用代码生成Anchors

In [ ]:
import math
import numpy as np
from src.mrcnn import course, config

conf = config.Config()

IMAGE_WIDTH  = conf.IMAGE_SHAPE[0]
IMAGE_HEIGHT = conf.IMAGE_SHAPE[1]

# 首先，我们计算FPN每一层特征图的尺寸
backbone_shapes = []
for stride in conf.BACKBONE_STRIDES:
    width  = math.ceil(IMAGE_WIDTH / stride)
    height = math.ceil(IMAGE_HEIGHT / stride)
    backbone_shapes.append([width, height])
backbone_shapes = np.array(backbone_shapes)

anchors = []
for i in range(backbone_shapes.shape[0]):
    scale = conf.RPN_ANCHOR_SCALES[i]
    shape = backbone_shapes[i]
    feature_stride = conf.BACKBONE_STRIDES[i]
    
    scales, ratios = np.meshgrid(np.array(scale), np.array(conf.RPN_ANCHOR_RATIOS))

    scales = scales.flatten()
    ratios = ratios.flatten()
    
    heights = scales / np.sqrt(ratios)
    widths = scales * np.sqrt(ratios)
    
    shifts_y = np.arange(0, shape[0], conf.RPN_ANCHOR_STRIDE) * feature_stride
    shifts_x = np.arange(0, shape[1], conf.RPN_ANCHOR_STRIDE) * feature_stride
    shifts_x, shifts_y = np.meshgrid(shifts_x, shifts_y)
    
    box_widths, box_centers_x = np.meshgrid(widths, shifts_x)
    box_heights, box_centers_y = np.meshgrid(heights, shifts_y)
    
    box_centers = np.stack(
        [box_centers_y, box_centers_x], axis=2).reshape([-1, 2])
    box_sizes = np.stack([box_heights, box_widths], axis=2).reshape([-1, 2])
    
    boxes = np.concatenate([box_centers - 0.5 * box_sizes,
                            box_centers + 0.5 * box_sizes], axis=1)
    
    anchors.append(boxes)
anchors = np.concatenate(anchors, axis=0)

# 打印anchors的维度
print(anchors.shape)

在上面计算Anchor的过程中，我们针对每种尺寸的特征图上的单个像素生成Anchor，以P2为例，P2的stride为4，即P2特征图的一个像素对应原图的4x4像素区域，我们对FPN上的每一层加入一个缩放参数$scale$，$scale_2$为32，以32为边长来生成Anchor，则P2位置在(0, 0)的像素点，以像素点坐标(0, 0)为中心，生成宽为$32 \times \sqrt{2}=45.3$，高为$32 \div \sqrt{2}=22.6$的Anchor。然后生成宽高都是32的Anchor，再生成宽高分别为22.6，45.3的Anchor，满足比例[0.5, 1, 2]。我们用四元组保存Anchor，保存Anchor两个对角的坐标点。以P2的第一个Anchor为例，中心点为(0, 0)，宽为45.3，高为22.6，则中心点个减去宽高的一半，就得到左下角点的坐标：

$$(0, 0) - (\frac{45.3}{2}, \frac{22.6}{2}) = (-22.6, -11.3)$$

同理，右上角的坐标为：

$$(0, 0) + (\frac{45.3}{2}, \frac{22.6}{2}) = (22.6, 11.3)$$


第一个Anchor的四元组就是：

$$(-22.6, -11.3, 22.6, 11.3)$$

最后，我们将所有Anchor合并到同一个数组中，因此上面代码中，变量anchors的shape为(261888, 4)。下面，我们显示前3个anchor，查看3个anchor之间的比例关系。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline


def draw_anchor(anchor, color='b'):
    bottom_left = anchor[:2]
    top_right = anchor[2:]
    
    currentAxis = plt.gca()
    currentAxis.set_aspect('equal')
    
    start_x = bottom_left[0]
    start_y = bottom_left[1]
    width = top_right[0] - start_x
    height = top_right[1] - start_y

    
    rect = patches.Rectangle((start_x, start_y), width, height,linewidth=2,edgecolor=color,facecolor='none')
    currentAxis.add_patch(rect)

# setup pyplot
plt.xlim([-30, 30])
plt.ylim([-30, 30])
plt.grid(True)

# draw & show anchor
draw_anchor(anchors[0], 'r')
draw_anchor(anchors[1], 'r')
draw_anchor(anchors[2], 'r')
print(anchors[0])
print(anchors[1])
print(anchors[2])

plt.show()

我们以3个anchor为一组，显示3组、共计9个anchor，查看anchor位置的移动过程。

In [ ]:
# setup pyplot
plt.xlim([-40, 40])
plt.ylim([-40, 40])
plt.grid(True)

colors = ['r', 'g', 'b']
# draw & show anchor
for i in range(9):
    c = colors[int(i / 3)] 
    draw_anchor(anchors[i], c)
plt.show()

以上就是Anchor的生成过程，注意生成的Anchor有可能超出图片的边界，如上述Anchor中就有对角点坐标都小于0的情况，在后面的过程中，会针对种情况进行处理。

我们需要将FPN中的特征图输入到RPN网络，对其进行卷及操作，以获得训练Anchor的结构。Anchor训练主要包括两个部分：

- Anchor的前景、背景分数
- Anchor的包围框信息

为此，我们先逐层处理特征图，首先将特征图进行3x3的卷积，然后在此卷积的基础上，分别进行两个1x1卷积，用于上述两部分信息的训练。其中一个卷积的深度为3x2=6，即表示每个像素上有3个Anchor，每个Anchor有2个分数（表示前景和背景的概率），这一部分在输入到softmax层进行激活，构成一个二分类问题，前景概率+背景概率为1。另一个卷积的深度为3x4=12，即每个像素上有3个Anchor，每个Anchor的包围框信息为一个四元组，四元组的内容为$[x, y, log(w), log(h)]$，然后该卷积再输入到一个linear激活函数。特征图在RPN中的处理过程如下图所示：

![featuremap_in_rpn](img/featuremap_in_rpn.jpg)

接下来，我们定义rpn_graph函数，对FPN中的每一层特征图进行上述处理：

In [ ]:
import tensorflow as tf
import keras.models as KM
import keras.layers as KL


def rpn_graph(feature_map, anchors_per_location, anchor_stride):
    # Shared convolutional base of the RPN
    shared = KL.Conv2D(512, (3, 3), padding='same', activation='relu',
                       strides=anchor_stride,
                       name='rpn_conv_shared')(feature_map)

    # Anchor Score. [batch, height, width, anchors per location * 2].
    x = KL.Conv2D(2 * anchors_per_location, (1, 1), padding='valid',
                  activation='linear', name='rpn_class_raw')(shared)

    # Reshape to [batch, anchors, 2]
    rpn_class_logits = KL.Lambda(
        lambda t: tf.reshape(t, [tf.shape(t)[0], -1, 2])
        , name='rpn_class_logits'
    )(x)

    # Softmax on last dimension of BG/FG.
    rpn_probs = KL.Activation(
        "softmax", name="rpn_class_xxx")(rpn_class_logits)

    # Bounding box refinement. [batch, H, W, anchors per location * depth]
    # where depth is [x, y, log(w), log(h)]
    x = KL.Conv2D(anchors_per_location * 4, (1, 1), padding="valid",
                  activation='linear', name='rpn_bbox_pred')(shared)

    # Reshape to [batch, anchors, 4]
    rpn_bbox = KL.Lambda(lambda t: tf.reshape(t, [tf.shape(t)[0], -1, 4])
                         , name='rpn_bbox'
                         )(x)

    return [rpn_class_logits, rpn_probs, rpn_bbox]

def build_rpn_model(anchor_stride, anchors_per_location, depth):
    input_feature_map = KL.Input(shape=[None, None, depth], name="input_rpn_feature_map")
    outputs = rpn_graph(input_feature_map, anchors_per_location, anchor_stride)
    return KM.Model([input_feature_map], outputs, name="rpn_model")

构造RPN模型并显示其结构

In [ ]:
RPN_FEATURE_STRIDE = 1
RPN_PER_LOCATION = 3
TOP_DOWN_PYRAMID_SIZE = 256
rpn_model = build_rpn_model(RPN_FEATURE_STRIDE, RPN_PER_LOCATION, TOP_DOWN_PYRAMID_SIZE)

from keras.utils import plot_model

plot_model(rpn_model, to_file='rpn_model.png')

![rpn_mode](rpn_model.png)

接下来，我们将FPN中每一层都输入到RPN模型中，并返回3个tensor，分别表示特征图Anchor的二分类，前景背景的分数（经过softmax激活）和包围框修正信息。然后将每一层的这3个tensor合并到一起，相当于把所有特征图处理的output进行了扁平化处理，存放在一个大的一维数组中。

In [ ]:
import keras.layers as KL
input_layer = KL.Input(shape=[1024, 1024, 3]) # 假设我们将图片尺寸统一为1024x1024
rpn_feature_maps = course.backbone(input_layer, conf)
# rpn_feature_maps包含了[P2, P3, P4, P5, P6]

# 对每一层都进行RPN网络的处理
layer_outputs = []  # list of lists
for p in rpn_feature_maps:
    layer_outputs.append(rpn_model([p]))

# 将输出结果中各层的rpn_class_logis, rpn_class和rpn_bbox，拼接到一个数组中
output_names = ["rpn_class_logits", "rpn_class", "rpn_bbox"]
outputs = list(zip(*layer_outputs))
    
outputs = [KL.Concatenate(axis=1, name=n)(list(o))
           for o, n in zip(outputs, output_names)]

rpn_class_logits, rpn_class, rpn_bbox = outputs

In [ ]:
print(rpn_class_logits.shape)
print(rpn_class.shape)
print(rpn_bbox.shape)

### Anchor前景背景的判定

Anchor生成后，我们首先要判定Anchor是否框住了图片中的物体，可以通过Anchor与Ground Truth物体边框的重叠程度来判定。这里我们引入IoU的概念。IoU即Intersection Over Union，也就是两个集合的交集与并集的商。假设A与B为两个集合，则：

$$IoU(A, B) = \frac {A \cap B} { A \cup B}$$

从图形的角度来看，计算IoU就是计算两个矩形框重叠部分面积占两者叠加面积的比值，如下图所示：
![iou](img/iou.jpg)

显然，IoU取值范围是$[0, 1]$，当两个矩形框没有任何重叠时，IoU为0，两者完全重合时，IoU为1，IoU的值越大，说明两个矩形框的重合度越高。在处理Anchor时，我们设置两个阈值，一般为0.3和0.7，当IoU大于0.7时，认为Anchor框住了某个物体，是前景，标记为正样本；IoU小于0.3时，认为Anchor没有框住物体，是背景，标记为负样本；IoU介于0.3和0.7的Anchor，认为是比较模糊的样本，不用于训练。这样，我们进行前景背景判定时，使用标记好的正负样本对Anchor进行二分类的训练，以softmax函数激活，得到Anchor为前景和背景的分数，也即概率，两者之和为1。

### Anchor锚框修正信息

找到Anchor的正负样本之后，我们还要计算样本与Ground Truth之间的偏移量，主要通过两者的中心坐标点和宽高之间的偏差来计算。假设Ground Truth物体边框的中心点坐标和宽高分别为：

$$(x, y), (w, h)$$

Anchor中心点和宽高分别为：

$$(x_a, y_a), (w_a, h_a)$$

我们按照下面的方式来计算偏移量：

$$\Delta x = \frac{x - x_a} {w_a}$$

$$\Delta y = \frac{y - y_a} {h_a}$$

$$\Delta w = log(\frac{w - w_a} {w_a})$$

$$\Delta h = log(\frac{h - h_a} {h_a})$$

这样，Anchor的修正信息也就可以用四元组$[\Delta x, \Delta y, \Delta w, \Delta h]$来保存。在生成数据的过程中，我们用上述方法来筛选样本，并用于Anchor分数、边框偏移量的训练。

### RoI

RoI，即Region of Interest，字面上的意思为"感兴趣区域"，也就是我们认为有可能是框住物体的区域，我们按照一定的规则对Anchor进行处理，将筛选后的Anchor作为RoI。

Anchor经过前面的训练，得到前景背景分数和边框偏移量。我们按照前景分数对Anchor进行排序，得到前景分数最高的若干个Anchor，这里我们需要设置一个用于筛选的Anchor数量，假设为2000，如果高分Anchor数量不足，则用0来填充，满足2000个Anchor的数量。然后，我们将训练中得到的边框修正信息累加到Anchor原始边框中，得到相对较为准确的修正框，然后使用NMS(Non-Max Supression，非极大值抑制)方法对修正框进行一次筛选，最终得到最为匹配真实物体边框的若干个修正框，即为RoI。

**NMS解决了什么问题？**

在我们得到的前景分数较高的Anchor中，往往有很多Anchor都满足IoU阈值，并且框住了同一个物体。这时，我们为了减小计算量，并获得更准确的结果，需要对这些Anchor进行处理，剔除重复，得到与真实框最接近的Anchor。下面我们简单介绍NMS处理的过程，假设我们有Anchor的集合
$$S=\{a, b, c, d, e, f\}$$

$S$中的元素按照分数从高到低排序，取出得分最高的元素$a$，放到另一个集合$T$中，则：

$$S=\{b, c, d, e, f\}, T=\{a\}$$

从$S$中，找到与$a$的IoU大于某个阈值的元素，假设为$b, c$，将$b, c$丢弃，这时：

$$S=\{d, e, f\}, T=\{a\}$$

在从$S$中取出得分最高的$d$放入$T$中，然后从剩下的$\{e，f\}$剔除与$d$的IoU高于阈值的元素，假设为$e, f$，将其丢弃。这时候$S$变为空集合，NMS过程结束，集合$T=\{a, d\}$就是我们筛选后的结果。

### 生成head网络数据

得到RoI后，我们生成head网络所需的数据。首先，RoI与Ground Truth box的IoU，这一步IoU的阈值一般设置为0.5，得到正负样本。然后，针对正样本，需要赋给下列3个值：

1. 正样本RoI所框住的物体的分类，即class_id
2. 正样本RoI与Ground Truth box的偏移量
3. 正样本RoI的Mask信息

对于class_id，我们找到与其IoU最大的Ground Truth标注信息，将标注信息中的分类class_id赋给RoI，Mask信息也赋给RoI，然后计算Ground Truth box与RoI的偏移量，计算方式与Anchor box偏移量计算一致。这样RoI正样本中，就包含了具体某个分类的信息、Mask信息和偏移量信息。我们将这写带有训练信息的RoI成为Proposal，Proposal与backbone网络中生成特征图传入RoI Align层，进行进一步的操作。